In [4]:
import gradio as gr
import csv

#Erick Santiago Rincon Rojas - 01251151002

escenarios = []
id_counter = 1

def calcular_cdt(monto, plazo):
    tasa = 0.001695 * plazo + 0.0983
    tasa_decimal = tasa / 100
    proyeccion = [{"mes": 0, "interes": 0, "saldo": monto}]
    saldo = monto
    for mes in range(1, plazo + 1):
        interes = saldo * tasa_decimal
        saldo += interes
        proyeccion.append({"mes": mes, "interes": interes, "saldo": saldo})
    return {
        "monto": monto, "plazo": plazo, "tasa": tasa,
        "saldo_final": saldo, "total_intereses": saldo - monto,
        "proyeccion": proyeccion
    }

def format_cop(valor):
    return f"$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

def actualizar_tabla():
    if not escenarios:
        return []
    return [[e["id"], format_cop(e["monto"]), e["plazo"],
             f"{e['tasa']:.4f}%", format_cop(e["saldo_final"]),
             format_cop(e["total_intereses"])] for e in escenarios]

def añadir_escenario(monto, plazo):
    global id_counter
    if monto is None or monto <= 0:
        return "⚠️ Monto debe ser mayor que cero", actualizar_tabla(), generar_proyecciones(), 0, 0
    if plazo is None or plazo < 1 or plazo != int(plazo):
        return "⚠️ Plazo debe ser entero >= 1", actualizar_tabla(), generar_proyecciones(), 0, 0

    cdt = calcular_cdt(monto, plazo)
    cdt["id"] = id_counter
    id_counter += 1
    escenarios.append(cdt)
    return f"✅ CDT #{cdt['id']} añadido", actualizar_tabla(), generar_proyecciones(), 0, 0

def eliminar_escenario(id_esc):
    if id_esc is None:
        return "⚠️ Ingresa un ID válido", actualizar_tabla(), generar_proyecciones()
    id_esc = int(id_esc)
    escenarios[:] = [e for e in escenarios if e["id"] != id_esc]
    return f"🗑️ CDT #{id_esc} eliminado", actualizar_tabla(), generar_proyecciones()

def actualizar_escenario(id_esc, monto, plazo):
    if id_esc is None:
        return "⚠️ Ingresa un ID válido", actualizar_tabla(), generar_proyecciones()
    if monto is None or monto <= 0:
        return "️ Monto debe ser mayor que cero", actualizar_tabla(), generar_proyecciones()
    if plazo is None or plazo < 1 or plazo != int(plazo):
        return "⚠️ Plazo debe ser entero >= 1", actualizar_tabla(), generar_proyecciones()

    id_esc = int(id_esc)
    cdt = next((e for e in escenarios if e["id"] == id_esc), None)
    if not cdt:
        return "❌ ID no encontrado", actualizar_tabla(), generar_proyecciones()

    nuevo_cdt = calcular_cdt(monto, plazo)
    nuevo_cdt["id"] = id_esc
    idx = escenarios.index(cdt)
    escenarios[idx] = nuevo_cdt
    return f"✅ CDT #{id_esc} actualizado", actualizar_tabla(), generar_proyecciones()

def generar_proyecciones():
    if not escenarios:
        return "Añade escenarios para ver las proyecciones"

    html = ""
    for e in escenarios:
        html += f"### 📊 CDT #{e['id']} — Monto: {format_cop(e['monto'])} | Plazo: {e['plazo']} meses | Tasa: {e['tasa']:.4f}%\n\n"
        html += "| Mes | Intereses | Saldo Final |\n"
        html += "|-----|-----------|-------------|\n"
        for p in e["proyeccion"]:
            html += f"| {p['mes']} | {format_cop(p['interes'])} | {format_cop(p['saldo'])} |\n"
        html += f"\n**Saldo final:** {format_cop(e['saldo_final'])} | **Total intereses:** {format_cop(e['total_intereses'])}\n\n---\n\n"
    return html

def exportar_escenarios_csv():
    if not escenarios:
        return "⚠️ No hay escenarios para exportar"
    with open("escenarios_cdt.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["ID", "Monto Inicial", "Plazo (meses)", "Tasa Mensual (%)", "Saldo Final", "Total Intereses"])
        for e in escenarios:
            writer.writerow([e["id"], e["monto"], e["plazo"],
                           f"{e['tasa']:.4f}", f"{e['saldo_final']:.2f}",
                           f"{e['total_intereses']:.2f}"])
    return "✅ Escenarios exportados a escenarios_cdt.csv"

def exportar_proyecciones_csv():
    if not escenarios:
        return "⚠️ No hay proyecciones para exportar"
    with open("proyecciones_cdt.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["ID Escenario", "Monto Inicial", "Plazo", "Tasa (%)", "Mes", "Intereses", "Saldo Final"])
        for e in escenarios:
            for p in e["proyeccion"]:
                writer.writerow([
                    e["id"], e["monto"], e["plazo"], f"{e['tasa']:.4f}",
                    p["mes"], f"{p['interes']:.2f}", f"{p['saldo']:.2f}"
                ])
    return "✅ Proyecciones exportadas a proyecciones_cdt.csv"

with gr.Blocks(title="Proyección CDT") as app:
    gr.Markdown("# 📊 Simulador de CDT")

    with gr.Tabs():
        with gr.TabItem(" Escenarios"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### Nuevo escenario")
                    in_monto = gr.Number(label="Monto inicial", value=0)
                    in_plazo = gr.Number(label="Plazo (meses)", value=1, precision=0)
                    btn_add = gr.Button("➕ Añadir", variant="primary")

                    gr.Markdown("### Editar escenario")
                    in_id_edit = gr.Number(label="ID del escenario", value=0, precision=0)
                    edit_monto = gr.Number(label="Nuevo monto", value=0)
                    edit_plazo = gr.Number(label="Nuevo plazo (meses)", value=1, precision=0)
                    btn_update = gr.Button("✏️ Actualizar", variant="secondary")

                    gr.Markdown("### Eliminar escenario")
                    in_id_del = gr.Number(label="ID a eliminar", value=0, precision=0)
                    btn_del = gr.Button("🗑️ Eliminar", variant="stop")

                    msg = gr.Textbox(label="Estado", interactive=False)

                with gr.Column(scale=2):
                    gr.Markdown("### Tabla de escenarios")
                    tabla = gr.Dataframe(
                        headers=["ID", "Monto", "Plazo", "Tasa", "Saldo Final", "Total Intereses"],
                        label="Escenarios",
                        interactive=False,
                        wrap=True
                    )
                    btn_exp_esc = gr.Button("📤 Exportar escenarios CSV", variant="secondary")

        with gr.TabItem(" Proyecciones"):
            gr.Markdown("### Proyección mes a mes de todos los escenarios")
            proyeccion_md = gr.Markdown("Añade escenarios para ver las proyecciones")
            btn_exp_proy = gr.Button("📤 Exportar proyecciones CSV", variant="secondary")
            msg_proy = gr.Textbox(label="Estado", interactive=False)

    btn_add.click(fn=añadir_escenario, inputs=[in_monto, in_plazo],
                  outputs=[msg, tabla, proyeccion_md, in_monto, in_plazo])
    btn_update.click(fn=actualizar_escenario, inputs=[in_id_edit, edit_monto, edit_plazo],
                     outputs=[msg, tabla, proyeccion_md])
    btn_del.click(fn=eliminar_escenario, inputs=[in_id_del],
                  outputs=[msg, tabla, proyeccion_md])
    btn_exp_esc.click(fn=exportar_escenarios_csv, outputs=[msg])
    btn_exp_proy.click(fn=exportar_proyecciones_csv, outputs=[msg_proy])

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d5755c63fba86eae0d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
